In [1]:
import pandas as pd 
import numpy as np 
import parquet

# maps
import geopandas as gpd
import json
import branca.colormap as cm
import folium
from folium.plugins import TimestampedGeoJson

# visualization
import plotly.graph_objects as go
import seaborn as sns 
import matplotlib.pyplot as plt 



In [ ]:
from pathlib import Path

DATA_DIR = Path('data')


trips = pd.read_parquet(DATA_DIR / 'trips', columns=['start_station_id', 'year'])
trips['year'] = trips['year'].astype(int)  # Hive partition column reads back as category

stations = pd.read_parquet(DATA_DIR / 'stations')[['station_id', 'name', 'neighbourhood', 'ward']]

df = trips.merge(
    stations.rename(columns={
        'station_id': 'start_station_id', 'name': 'start_station_name',
        'neighbourhood': 'start_neighbourhood', 'ward': 'start_ward',
    }),
    on='start_station_id', how='left',
)

print(df.shape)
print(df.columns.tolist())
print("unmatched neighbourhood:", df['start_neighbourhood'].isna().sum())

(44638688, 5)
['start_station_id', 'year', 'start_station_name', 'start_neighbourhood', 'start_ward']
unmatched neighbourhood: 28119


In [3]:
wards = gpd.read_parquet(DATA_DIR / 'wards' / 'city=toronto' / 'part.parquet')

In [4]:
print(wards[['ward']].sort_values('ward'))
print(wards.crs)

                        ward
4          Beaches-East York
20                 Davenport
19           Don Valley East
18          Don Valley North
11           Don Valley West
17         Eglinton-Lawrence
16          Etobicoke Centre
14           Etobicoke North
15       Etobicoke-Lakeshore
0   Humber River-Black Creek
13        Parkdale-High Park
9         Scarborough Centre
7          Scarborough North
5      Scarborough Southwest
10     Scarborough-Agincourt
8      Scarborough-Guildwood
6     Scarborough-Rouge Park
24         Spadina-Fort York
21            Toronto Centre
23          Toronto-Danforth
12        Toronto-St. Paul's
3        University-Rosedale
2                 Willowdale
1                York Centre
22         York South-Weston
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": 

## Stations

In [23]:
df.columns

Index(['start_station_id', 'year', 'start_station_name', 'start_neighbourhood',
       'start_ward'],
      dtype='str')

In [7]:
# stations_yearly = stations_lagged.groupby('year', as_index=False)['nr_stations'].sum()
stations_yearly = df.groupby('year')['start_station_id'].nunique().reset_index(name='nr_stations')

In [103]:
fig = go.Figure(go.Scatter(
    x=stations_yearly['year'],
    y=stations_yearly['nr_stations'],
    mode='lines+markers+text',
    text=[f"{v}" for v in stations_yearly['nr_stations']],
    textposition='top center',
    line=dict(color="#FF6B4A", width=3),
    marker=dict(color='#FF6B4A', size=10, line=dict(color='white', width=2)),
))

fig.update_layout(
    title=dict(text='Stations Per Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black',tickvals=stations_yearly['year']),
    yaxis=dict(title='Stations', showgrid=False, showline=True, linecolor='black', tickformat='.2s'),
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=550,
)

fig

In [ ]:
# calculate change in number stations per ward:
stations_lagged = (
    df.groupby(['year', 'start_ward'])['start_station_id']
    .nunique()
    .reset_index(name='nr_stations')
    .sort_values(['start_ward', 'year'])  
    .assign(lag=lambda d: d.groupby('start_ward')['nr_stations'].shift(1))
)


In [24]:
stations_lagged

,year,start_ward,nr_stations,lag
0,2016,Beaches-East York,7,NaN
8,2017,Beaches-East York,7,7.0
18,2018,Beaches-East York,16,7.0
30,2019,Beaches-East York,32,16.0
43,2020,Beaches-East York,42,32.0
...,...,...,...,...
99,2022,York South-Weston,4,4.0
120,2023,York South-Weston,23,4.0
143,2024,York South-Weston,27,23.0
168,2025,York South-Weston,32,27.0


## Trips

In [13]:
# trips dynamics year on year by ward
df_lagged = (
    df.groupby(['year', 'start_ward'])
    .size()
    .reset_index(name='nr_trips')
    .sort_values(['start_ward', 'year'])  # Sorted by ward then year for proper shifting
    .assign(lag=lambda d: d.groupby('start_ward')['nr_trips'].shift(1))
)


In [14]:
df_lagged.groupby('year', as_index=False)['nr_trips'].sum()

,year,nr_trips
0,2016,318933
1,2017,1348106
2,2018,1922954
3,2019,2439399
4,2020,2909386
5,2021,3569606
6,2022,4615252
7,2023,5708745
8,2024,6944552
9,2025,7807208


In [15]:
trips_tbl = (
    df_lagged
    .assign(delta=lambda d: round(100.0 * (d['nr_trips'] - d['lag']) / d['nr_trips'], 1))
    .pivot_table(columns='year', values='nr_trips', index='start_ward') 
    .reset_index()
    .rename_axis(None, axis=1) 
    .sort_values(by=2025, ascending=False) 
)

In [106]:
trips

,start_station_id,year
0,7100,2016
1,7097,2016
2,7100,2016
3,7033,2016
4,7059,2016
...,...,...
44638683,7382,2026
44638684,8026,2026
44638685,7236,2026
44638686,7944,2026


In [109]:
trips_yearly = (trips
                    .groupby('year')['start_station_id']
                    .size()
                    .reset_index(name='nr_trips')
                    )

In [117]:
fig = go.Figure(go.Scatter(
    x=trips_yearly['year'],
    y=trips_yearly['nr_trips'],
    mode='lines+markers+text',
    text=[f"{v/1e6:.1f}M" for v in trips_yearly['nr_trips']],
    textposition='top center',
    line=dict(color="#2ad6b1", width=3),
    marker=dict(color='#2ad6b1', size=10, line=dict(color='white', width=2)),
))

fig.update_layout(
    title=dict(text='Trips Per Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black',tickvals=stations_yearly['year']),
    yaxis=dict(title='Trips', showgrid=False, showline=True, linecolor='black',  dtick=1000000, ticks='outside', ticklen=6, tickcolor='black'),
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=550,
)

fig

In [ ]:
years = sorted(df['year'].unique())

In [ ]:
ward_year_raw = {}
ward_year_log = {}
all_log_values = []

for year in years:
    trips_year = df[df['year'] == year]
    counts = trips_year.groupby('start_ward').size()
    ward_year_raw[year] = counts
    log_counts = np.log(counts)
    ward_year_log[year] = log_counts
    all_log_values.extend(log_counts.values)

vmin, vmax = min(all_log_values), max(all_log_values)
colormap = cm.linear.YlGn_09.scale(vmin, vmax)

features = []
for ward_name, geom in zip(wards['ward'], wards['geometry']):
    geom_json = geom.__geo_interface__
    for year in years:
        log_count = ward_year_log[year].get(ward_name, vmin)
        raw_count = int(ward_year_raw[year].get(ward_name, 0))
        color = colormap(log_count)
        features.append({
            "type": "Feature",
            "geometry": geom_json,
            "properties": {
                "times": [f"{year}-01-01"],
                "style": {
                    "color": "white",
                    "fillColor": color,
                    "fillOpacity": 0.85,
                    "weight": 1
                },
                "tooltip": f"{ward_name}: {raw_count:,} trips ({year})"
            }
        })

geo_timed = {"type": "FeatureCollection", "features": features}
print(len(features), "features built")

In [ ]:
m = folium.Map(
    location=[43.72, -79.42],
    zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
)

TimestampedGeoJson(
    geo_timed,
    period='P1Y',
    duration='P1Y',
    auto_play=False,
    loop=False,
    date_options='YYYY',
    transition_time=300,
).add_to(m)

colormap.caption = 'Trips started (log scale)'
colormap.add_to(m)

m.save('ward_map_timelapse.html')
print("saved ward_map_timelapse.html")

## MAPS


In [ ]:
trips_2025 = df[df['year'] == 2025]
ward_counts_2025 = trips_2025.groupby('start_ward').size()
print(ward_counts_2025.sort_values(ascending=False))
print("total wards with data:", len(ward_counts_2025), "/ 25")

In [ ]:
ward_counts_2025_log = np.log(ward_counts_2025)

In [ ]:
wards_out = wards.copy()
wards_out['trips_2025'] = wards_out['ward'].map(ward_counts_2025).fillna(0).astype(int)
wards_out['trips_2025_log'] = wards_out['ward'].map(ward_counts_2025_log).fillna(ward_counts_2025_log.min())
wards_out = wards_out[['ward', 'trips_2025', 'trips_2025_log', 'geometry']]
wards_out.to_file('/tmp/toronto_wards_trips2025.geojson', driver='GeoJSON')
print(wards_out.drop(columns='geometry').sort_values('trips_2025', ascending=False))

In [ ]:
import folium

m = folium.Map(
    location=[43.72, -79.42],
    zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
)

choropleth = folium.Choropleth(
    geo_data=wards_out,
    data=wards_out,
    columns=['ward', 'trips_2025_log'],  # color by log scale
    key_on='feature.properties.ward',
    fill_color='YlGn',
).add_to(m)

folium.GeoJsonTooltip(
    fields=['ward', 'trips_2025'],  # tooltip shows the actual trip count
    aliases=['Ward:', 'Trips (2025):'],
    localize=True
).add_to(choropleth.geojson)

In [36]:
m.save('ward_map.html')
print("saved")


saved


In [ ]:
print(sorted(df['year'].unique()))

In [ ]:
years = sorted(df['year'].unique())

for year in years:
    trips_year = df[df['year'] == year]
    ward_counts_year = trips_year.groupby('start_ward').size()
    ward_counts_year_log = np.log(ward_counts_year)

    wards_out = wards.copy()
    wards_out[f'trips_{year}'] = wards_out['ward'].map(ward_counts_year).fillna(0).astype(int)
   
    wards_out[f'trips_{year}_log'] = (
        wards_out['ward'].map(ward_counts_year_log).fillna(ward_counts_year_log.min())
    )
    wards_out = wards_out[['ward', f'trips_{year}', f'trips_{year}_log', 'geometry']]

    m = folium.Map(
        location=[43.72, -79.42],
        zoom_start=11,
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
        attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    )

    choropleth = folium.Choropleth(
        geo_data=wards_out,
        data=wards_out,
        columns=['ward', f'trips_{year}_log'],
        key_on='feature.properties.ward',
        fill_color='YlGn',
    ).add_to(m)

    folium.GeoJsonTooltip(
        fields=['ward', f'trips_{year}'],
        aliases=['Ward:', f'Trips ({year}):'],
        localize=True
    ).add_to(choropleth.geojson)

    m.save(f'ward_map_{year}.html')
    print(f"saved ward_map_{year}.html")

In [ ]:
import json
import branca.colormap as cm
from folium.plugins import TimestampedGeoJson

ward_year_counts = {}
all_log_values = []
for year in years:
    trips_year = df[df['year'] == year]
    counts = trips_year.groupby('start_ward').size()
    log_counts = np.log(counts)
    ward_year_counts[year] = log_counts
    all_log_values.extend(log_counts.values)

vmin, vmax = min(all_log_values), max(all_log_values)
colormap = cm.linear.YlGn_09.scale(vmin, vmax)
print("log-count range across all years:", vmin, vmax)

In [41]:
import inspect
print(folium.__version__)
print(inspect.signature(TimestampedGeoJson.__init__))


0.20.0
(self, data, transition_time=200, loop=True, auto_play=True, add_last_point=True, period='P1D', min_speed=0.1, max_speed=10, loop_button=False, date_options='YYYY-MM-DD HH:mm:ss', time_slider_drag_update=False, duration=None, speed_slider=True)


## STATIONS MAP (dot timelapse)

In [ ]:
stations_full = pd.read_parquet(DATA_DIR / 'stations')[['station_id', 'name', 'lat', 'lon']]
first_year = trips.groupby('start_station_id')['year'].min().rename('first_year')
total_trips = trips.groupby('start_station_id').size().rename('total_trips')


stations_tl = (
    stations_full.set_index('station_id')
    .join(first_year, how='inner')
    .join(total_trips, how='inner')
    .dropna(subset=['lat', 'lon'])
    .reset_index()
)
print(stations_tl.shape, '/', stations_full.shape[0], 'stations plotted')
print(stations_tl['first_year'].value_counts().sort_index())

(1192, 6) / 1202 stations plotted
first_year
2016    200
2017     65
2018     93
2019    107
2020    146
2021     18
2022     66
2023    151
2024     88
2025    177
2026     81
Name: count, dtype: int64


In [25]:

MIN_RADIUS_PX, MAX_RADIUS_PX = 3, 12
DOT_COLOR = "#FF6B4A"  
                        

scaled = np.sqrt(stations_tl['total_trips'])
scaled_norm = (scaled - scaled.min()) / (scaled.max() - scaled.min())
stations_tl['radius_px'] = MIN_RADIUS_PX + scaled_norm * (MAX_RADIUS_PX - MIN_RADIUS_PX)

station_features = []
for row in stations_tl.itertuples():
    station_features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(row.lon), float(row.lat)]},
        "properties": {

            "times": [f"{row.first_year}-01-01"],
            "icon": "circle",
            "iconstyle": {
                "fillColor": DOT_COLOR, "fillOpacity": 0.7,
                "stroke": True, "color": "white", "weight": 1,
                "radius": float(row.radius_px),  
            },
            "tooltip": row.name,
        },
    })


last_year = int(stations_tl['first_year'].max())
station_features.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [float(stations_tl['lon'].iloc[0]), float(stations_tl['lat'].iloc[0])]},
    "properties": {
        "times": [f"{last_year + 1}-01-01"],
        "icon": "circle",
        "iconstyle": {"radius": 0, "fillOpacity": 0, "opacity": 0},
    },
})

geo_stations_timed = {"type": "FeatureCollection", "features": station_features}
print(len(station_features), "station features built (incl. 1 padding dummy)")

1193 station features built (incl. 1 padding dummy)


In [ ]:
m = folium.Map(

    location=[43.70, -79.37],
    zoom_start=12,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
)

TimestampedGeoJson(
    geo_stations_timed,
    period='P1Y',
    duration=None, 
    auto_play=False,
    loop=False,
    date_options='YYYY',
    transition_time=300,
).add_to(m)

m.save('stations_map_timelapse.html')
print("saved stations_map_timelapse.html")

saved stations_map_timelapse.html


## Busiest stations and the most popular routes.

In [36]:
trips_routes = pd.read_parquet(DATA_DIR / 'trips', columns=['start_station_id', 'end_station_id'])

In [41]:
busiest_stations = (
    trips_routes['start_station_id'].value_counts().head(5)
    .rename_axis('station_id').reset_index(name='trips')
)
busiest_stations['name'] = busiest_stations['station_id'].map(stations_full.set_index('station_id')['name'])
print(busiest_stations)


   station_id   trips                                     name
0        7076  376095                  York St / Queens Quay W
1        7006  298031          Bay St / College St (East Side)
2        7016  286991  Bay St / Queens Quay W (Ferry Terminal)
3        7030  262270                  Bay St / Wellesley St W
4        7033  259606                            Union Station


In [39]:
route_pairs = (
    trips_routes[trips_routes['start_station_id'] != trips_routes['end_station_id']]
    .groupby(['start_station_id', 'end_station_id']).size()
    .sort_values(ascending=False).head(5)
)

In [46]:
# stations_full.set_index('station_id')['name']

In [45]:
name_lookup = stations_full.set_index('station_id')['name']
for (s, e), cnt in route_pairs.items():
    print(cnt, name_lookup.get(s, s), '->', name_lookup.get(e, e))

35003 Centre Island Ferry Dock -> Ward's Island Ferry Dock
19550 Ward's Island Ferry Dock -> Centre Island Ferry Dock
19076 Centre Island Ferry Dock -> Centre Island
18233 Centre Island -> Ward's Island Ferry Dock
17522 Centre Island -> Centre Island Ferry Dock


## Seasonality patterns and behaviour

In [80]:
times = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'start_time'], filters = [('year', 'in', [2024, 2025, 2026])])
times.head()

,trip_id,year,start_time
0,26682740,2024,2024-01-01 00:00:08
1,26682742,2024,2024-01-01 00:02:04
2,26682743,2024,2024-01-01 00:03:25
3,26682744,2024,2024-01-01 00:03:39
4,26682746,2024,2024-01-01 00:04:44


In [81]:
times = (times
    .assign(year_month = lambda d: d['start_time'].dt.to_period('M'),
            hour = lambda d: d.start_time.dt.hour)
    )

In [83]:
monthly = (times
    .groupby(['year_month'])['trip_id']
    .size()   
    .reset_index(name='nr_trips')
    )

In [91]:
# 2ad6b1
# FF6B4A
fig_monthly = go.Figure(go.Bar(
    x=monthly['year_month'].dt.to_timestamp(),
    y=monthly['nr_trips'],
    marker=dict(color="#2ad6b1"),
    opacity=0.9
    
))

fig_monthly.update_layout(
    title=dict(text='Trips by Month, 2024-2026', x=0.5),
    xaxis=dict(title='Month', showgrid=False, dtick='M12', tickformat='%Y'),
    yaxis=dict(title='Trips', showgrid=False, tickformat='.2s'),
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1100,
    height=500,
)

fig_monthly